# 🔁 06_orquestrador — Execução contínua do pipeline (polling)
**Squad 2 · Dupla 3**

Notebook "gerente": não processa dados. Em laço, chama os notebooks do pipeline, lê o resumo de cada um, registra cada ciclo em `metadata/`
e encerra sozinho quando o gerador para de enviar arquivos.

| Etapa | Conteúdo |
|---|---|
| 1 | Configuração e parâmetros do laço |
| 2 | Execução de uma etapa do pipeline |
| 3 | Laço de polling |
| 4 | Relatório da execução e conferência final da Bronze |

**Como cada notebook roda** (`MODO`, Etapa 1):

| Modo | Como funciona | Quando usar |
|---|---|---|
| `mesmo_processo` (padrão) | lê o código do `03`, `04` e `05` pela API do workspace e roda dentro deste notebook, cada um com suas próprias variáveis | sempre: cada ciclo leva segundos e o erro aparece com a mensagem real |
| `notebook_run` | `dbutils.notebook.run`: cada notebook num ambiente separado | só se o modo padrão não puder ler os notebooks; no Serverless, iniciar cada notebook pode levar minutos e o erro chega como "Workload failed" |

O pipeline é Bronze → Silver → Gold. A Bronze roda em todo ciclo; Silver e Gold só quando ela traz arquivo novo, no primeiro ciclo ou para refazer um ciclo que teve erro (`SEMPRE_RODAR_TUDO = True` volta a rodar tudo sempre).
Para parar antes do fim: botão **Cancel** (o registro em `metadata/` é salvo a cada ciclo).

**Uma execução por vez:** o laço só começa se conseguir criar a trava `metadata/ecommerce_pedidos/orquestrador_trava.json`. Dois orquestradores ao mesmo tempo leriam os mesmos arquivos novos e gravariam a Bronze em dobro (regra 2). Uma trava sem sinal de vida há mais de 30 min é de uma execução que caiu e é descartada.
Cada notebook chamado recebe o nome desta execução (`execucao`) e confere a trava: com o orquestrador ativo, `03`, `04` e `05` rodados **à mão** se recusam a processar.

## Etapa 1 — Configuração
**Faz:** carrega o `00_setup_config` (para o registro em `metadata/` e a conferência final) e define os parâmetros do laço.
**Valida:** se os parâmetros são coerentes.

In [0]:
VALIDACAO_COMPLETA = False

In [0]:
%run ./00_setup_config

# ⚙️ 00_setup_config — Configuração central
**Squad 2 · Dupla 3 · `ecommerce_pedidos`**

Prepara endereços, credenciais, conexões e funções usadas por todo o pipeline. **Não lê nem grava pedidos.**

Cada etapa **faz o trabalho e valida em seguida**: ✅ ok · ⚠️ aviso (segue) · ❌ erro (para e explica).

**Uso nos outros notebooks** (duas células; o `%run` fica sozinho):
```
VALIDACAO_COMPLETA = False
```
```
%run ./00_setup_config
```
Dependências em **Environment → Dependencies**: `azure-identity`, `azure-storage-file-datalake`.

## Etapa 1 — Ambiente e endereços
**Faz:** cria a função `checar` e guarda os nomes fixos do projeto.
**Valida:** se está no Databricks e se origem e destino são containers diferentes.

Modo de validação: RÁPIDA
✅ Rodando no Databricks
✅ Origem (raw) e destino (squad2) separados


## Etapa 2 — Credenciais
**Faz:** busca as 7 chaves no Secret Scope. Ficam só na memória (aparecem como `[REDACTED]` se impressas).
**Valida:** se o cofre existe, se nenhuma chave falta e se o host do SQL tem o formato certo.

✅ Cofre 'mercadata-ecommerce-pedidos' encontrado
✅ 7 credenciais carregadas (valores ocultos)
✅ SQL_HOST no formato do Azure SQL


...

✅ url_origem OK
✅ tabela_delta_existe OK (Bronze já existe)
✅ ler_metadata OK (None para anotação inexistente)
✅ versao_delta OK (Bronze na versão 43)
✅ trava_ativa OK (orquestrador parado)


## Etapa 6 — Funções do SQL Server
**Faz:** cria as funções de gravar e consultar o banco. Gravação pelo conector `sqlserver` (o `jdbc` é bloqueado no Serverless); leitura pelo `jdbc`.
**Valida:** se toda tabela recebe o schema `squad2`.

✅ Tabelas SQL vão para o schema 'squad2'


## Etapa 7 — Teste completo (só ao rodar este notebook direto)
**Faz e valida:** varre o `raw` · lê o micro-lote mais recente e confere colunas e tipos (regra 1) · grava e relê em `metadata/` · `SELECT 1` no SQL Server · confere se o schema `squad2` existe no banco.
No modo rápido (`%run`), é pulada. No fim, limpa as variáveis temporárias.

Teste completo pulado (modo rápido).
⚙️ 00_setup_config pronto para uso


In [0]:
# Tudo o que o 00_setup_config definiu (funções, caminhos, spark, dbutils): é o ponto de partida de cada notebook rodado no modo mesmo_processo
BASE_NOTEBOOKS = {k: v for k, v in globals().items() if not k.startswith("_")}

import time
from pyspark.sql import functions as F

PIPELINE = ["./03_ingestao_bronze", "./04_ingestao_silver", "./05_ingestao_gold"]   # ordem de execução
MODO = "mesmo_processo"                  # "mesmo_processo": roda o código dos notebooks aqui dentro · "notebook_run": dbutils.notebook.run
INTERVALO_S = 60                         # espera entre ciclos
MAX_CICLOS_SEM_DADOS = 5                 # encerra após N ciclos seguidos sem arquivo novo (gerador parado)
SEMPRE_RODAR_TUDO = False                # False: Silver e Gold só rodam quando a Bronze traz novidade (ou para refazer um ciclo com erro)
MAX_CICLOS = 180                         # limite de segurança (~3 h com intervalo de 60 s)
TIMEOUT_S = 900                          # tempo máximo de cada notebook chamado (só no modo notebook_run)
INICIO = datetime.now(timezone.utc)
ANOTACAO_LOG = f"orquestrador_{INICIO:%Y%m%d_%H%M%S}"   # metadata/ecommerce_pedidos/orquestrador_AAAAMMDD_HHMMSS.json

TRAVA = TRAVA_ORQUESTRADOR               # metadata/ecommerce_pedidos/orquestrador_trava.json (nome e validade no 00_setup_config)


def ler_trava():
    """Conteúdo da trava ({} se não existe ou ainda está vazia, no instante entre a criação e a escrita)."""
    try:
        return ler_metadata(TRAVA) or {}
    except ValueError:
        return {}


def travar():
    """Cria a trava de forma atômica. Devolve None se conseguiu, ou o dono da trava ativa."""
    from azure.core import MatchConditions
    from azure.core.exceptions import ResourceExistsError, ResourceModifiedError
    cliente = fs_squad.get_file_client(caminho_metadata(TRAVA))
    for _ in range(2):
        try:
            # If-None-Match: * -> o Azure só cria se o arquivo não existir; entre dois processos, só um consegue
            cliente.create_file(match_condition=MatchConditions.IfMissing)
            renovar_trava()
            return None
        except (ResourceExistsError, ResourceModifiedError):
            # Idade pelo horário do próprio Azure: a trava é regravada a cada ciclo, então last_modified é o sinal de vida
            idade = datetime.now(timezone.utc) - cliente.get_file_properties().last_modified
            if idade.total_seconds() < TRAVA_EXPIRA_MIN * 60:
                return ler_trava() or {"execucao": "iniciando", "sinal_de_vida_utc": "agora"}
            cliente.delete_file()                # trava abandonada: descarta e tenta de novo
    return {"execucao": "desconhecida"}


def renovar_trava():
    escrever_metadata(TRAVA, {"execucao": ANOTACAO_LOG, "sinal_de_vida_utc": datetime.now(timezone.utc).isoformat()})


def destravar():
    if ler_trava().get("execucao") == ANOTACAO_LOG:   # só libera a própria trava
        fs_squad.get_file_client(caminho_metadata(TRAVA)).delete_file()


# ------------------------------ validação ------------------------------
checar(PIPELINE and PIPELINE[0].endswith("bronze"), f"Pipeline: {' → '.join(PIPELINE)}", "A primeira etapa do pipeline deve ser a Bronze")
checar(INTERVALO_S > 0 and 0 < MAX_CICLOS_SEM_DADOS <= MAX_CICLOS, "Parâmetros do laço coerentes", "Parâmetros do laço inválidos")
print(f"Intervalo {INTERVALO_S} s · para após {MAX_CICLOS_SEM_DADOS} ciclos sem dados ou {MAX_CICLOS} ciclos · registro: {caminho_metadata(ANOTACAO_LOG)}")

✅ Pipeline: ./03_ingestao_bronze → ./04_ingestao_silver → ./05_ingestao_gold
✅ Parâmetros do laço coerentes
Intervalo 60 s · para após 5 ciclos sem dados ou 180 ciclos · registro: metadata/ecommerce_pedidos/orquestrador_20261008_194726.json


## Etapa 2 — Execução de uma etapa
**Faz:** prepara a função que roda um notebook do pipeline e devolve o resumo JSON que ele publicou, ou o erro. Uma falha nunca derruba o laço.

- **`mesmo_processo`:** lê uma vez, pela API do workspace, as células de código do `03`, `04` e `05` (sem o `%run`, porque o `00_setup_config` já está carregado aqui).
  Cada notebook roda num dicionário próprio de variáveis, com o parâmetro `execucao` desta execução; o `dbutils.notebook.exit` vira o resumo,
  os prints ficam guardados e só aparecem se der erro, junto com a mensagem e o traceback.
- **`notebook_run`:** `dbutils.notebook.run`; quando o Databricks devolve só "Workload failed", a causa vem da anotação `ultimo_erro` gravada pelo `checar()`.

**Valida:** o modo é conhecido; no `mesmo_processo`, os três notebooks foram lidos e são a versão atual (têm `garantir_execucao_unica` e o resumo final).

In [0]:
import base64, contextlib, io, os, traceback


class _Saida(Exception):
    """dbutils.notebook.exit(...) de um notebook rodado aqui dentro: encerra a execução dele e carrega o resumo."""


class _NotebookLocal:
    def exit(self, valor):
        raise _Saida(valor)


class _WidgetsLocal:
    """Parâmetros do notebook rodado aqui dentro (o mesmo papel dos widgets que o dbutils.notebook.run preenche)."""
    def __init__(self, parametros):
        self.parametros = dict(parametros)

    def text(self, nome, padrao, *args, **kwargs):
        self.parametros.setdefault(nome, padrao)

    def get(self, nome):
        return self.parametros[nome]


class _DbutilsLocal:
    """dbutils entregue ao notebook rodado aqui dentro: exit e widgets locais; secrets, fs etc. continuam os de verdade."""
    def __init__(self, parametros):
        self.notebook, self.widgets = _NotebookLocal(), _WidgetsLocal(parametros)

    def __getattr__(self, nome):
        return getattr(dbutils, nome)


def pasta_deste_notebook():
    """Pasta do orquestrador no workspace (o Databricks abre cada notebook na pasta dele: /Workspace/...)."""
    if os.getcwd().startswith("/Workspace/"):
        return os.getcwd().removeprefix("/Workspace")
    return dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get().rsplit("/", 1)[0]


def carregar_codigo(notebook):
    """Células de código do notebook, lidas pela API do workspace, sem as mágicas (%run)."""
    from databricks.sdk import WorkspaceClient
    from databricks.sdk.service.workspace import ExportFormat
    exportado = WorkspaceClient().workspace.export(f"{PASTA_WORKSPACE}/{notebook.removeprefix('./')}", format=ExportFormat.JUPYTER).content
    celulas = json.loads(base64.b64decode(exportado))["cells"]
    fontes = ["".join(c["source"]) if isinstance(c["source"], list) else c["source"] for c in celulas if c["cell_type"] == "code"]
    return [f for f in fontes if f.strip() and not f.lstrip().startswith("%")]


def causa_do_erro(texto):
    """O erro de dbutils.notebook.run vem embrulhado (WorkflowException); procura a parte que explica a causa real."""
    for marca in ("❌", "AssertionError", "AnalysisException", "Error:", "Exception:"):
        posicao = texto.rfind(marca)
        if posicao >= 0:
            return texto[posicao:posicao + 300].strip()
    return texto[-300:].strip()


def executar(notebook):
    """Roda um notebook do pipeline e devolve {'status', 'duracao_s', 'resumo' ou 'erro'}."""
    inicio = time.time()
    if MODO == "notebook_run":
        try:
            resumo = json.loads(dbutils.notebook.run(notebook, TIMEOUT_S, {"execucao": ANOTACAO_LOG}) or "{}")   # execucao: a trava é desta execução
            return {"status": "OK", "duracao_s": round(time.time() - inicio, 1), "resumo": resumo}
        except Exception as erro:            # o erro fica registrado; o próximo ciclo tenta de novo
            texto, causa = str(erro), causa_do_erro(str(erro))
            anotado = ler_metadata("ultimo_erro") or {}
            if anotado.get("momento_utc", "") >= datetime.fromtimestamp(inicio, timezone.utc).isoformat():
                causa = anotado["mensagem"]
            return {"status": "ERRO", "duracao_s": round(time.time() - inicio, 1), "erro": causa, "erro_completo": f"{causa}\n{texto[-1500:]}"}

    # mesmo_processo: variáveis próprias para cada notebook (nada vaza de um para o outro) e prints guardados
    variaveis = {**BASE_NOTEBOOKS, "dbutils": _DbutilsLocal({"execucao": ANOTACAO_LOG}), "display": lambda *a, **k: None}
    saida = io.StringIO()
    try:
        with contextlib.redirect_stdout(saida):
            for celula in CODIGO[notebook]:
                exec(compile(celula, notebook, "exec"), variaveis)
        resumo = {}
    except _Saida as fim:
        resumo = json.loads(fim.args[0] or "{}")
    except Exception as erro:                # o erro fica registrado; o próximo ciclo tenta de novo
        causa = str(erro) if isinstance(erro, AssertionError) else f"{type(erro).__name__}: {erro}"
        return {"status": "ERRO", "duracao_s": round(time.time() - inicio, 1), "erro": causa[:300],
                "erro_completo": f"{causa[:1000]}\n--- últimas linhas do notebook ---\n{saida.getvalue()[-1500:]}"
                                 f"\n--- traceback ---\n{traceback.format_exc()[-2000:]}"}
    return {"status": "OK", "duracao_s": round(time.time() - inicio, 1), "resumo": resumo}


CODIGO, erro_leitura = {}, None
if MODO == "mesmo_processo":
    PASTA_WORKSPACE = pasta_deste_notebook()
    try:
        CODIGO = {nb: carregar_codigo(nb) for nb in PIPELINE}
    except Exception as erro:
        erro_leitura = f"{type(erro).__name__}: {str(erro)[:200]}"

# ------------------------------ validação ------------------------------
checar(MODO in ("mesmo_processo", "notebook_run"), f"Modo de execução: {MODO}", "MODO deve ser 'mesmo_processo' ou 'notebook_run'")
if MODO == "mesmo_processo":
    checar(erro_leitura is None, f"Notebooks lidos de {PASTA_WORKSPACE}",
           f"Não consegui ler os notebooks em {PASTA_WORKSPACE} ({erro_leitura}). "
           "Confira se o databricks-sdk está em Environment → Dependencies, ou use MODO = 'notebook_run'")
    for nb, celulas in CODIGO.items():
        texto = "\n".join(celulas)
        checar("dbutils.notebook.exit" in texto and "garantir_execucao_unica" in texto, f"{nb}: {len(celulas)} células de código",
               f"{nb}: não é a versão atual (sem garantir_execucao_unica ou sem o resumo final)")
checar(callable(executar), "Função de execução pronta", "Função de execução indisponível")

✅ Modo de execução: mesmo_processo
✅ Notebooks lidos de /Users/eduardogeraldoo@gmail.com/5-squad2_ecommerce_real_time - Claude
✅ ./03_ingestao_bronze: 8 células de código
✅ ./04_ingestao_silver: 8 células de código
✅ ./05_ingestao_gold: 11 células de código
✅ Função de execução pronta


## Etapa 3 — Laço de polling
**Faz:** a cada ciclo, roda a Bronze e, se houver novidade, a Silver e a Gold; imprime uma linha de acompanhamento e salva o histórico em `metadata/`.
Se uma etapa falhar, as seguintes do ciclo não rodam (dependem dela), e o próximo ciclo refaz tudo. Conta os ciclos sem arquivo novo e encerra no limite.
**Valida:** a trava foi obtida (nenhum outro orquestrador rodando); cada ciclo é registrado; erros aparecem na linha do ciclo e no relatório da Etapa 4.

In [0]:
historico, ciclo, sem_dados = [], 0, 0
refazer = True                           # o primeiro ciclo sempre roda tudo, para colocar o pipeline em dia
erros_mostrados = {}                     # notebook -> último erro já impresso por completo
outro = travar()
checar(outro is None, f"Trava obtida: só esta execução processa ({ANOTACAO_LOG})",
       f"Outro orquestrador está rodando ({outro and outro.get('execucao')}, sinal de vida {outro and outro.get('sinal_de_vida_utc')}): "
       "espere ele terminar ou pare o outro antes")
print(f"Início: {INICIO:%Y-%m-%d %H:%M:%S} UTC")
try:
    while ciclo < MAX_CICLOS and sem_dados < MAX_CICLOS_SEM_DADOS:
        ciclo += 1
        renovar_trava()
        registro = {"ciclo": ciclo, "inicio_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}
        registro[PIPELINE[0]] = executar(PIPELINE[0])          # a Bronze sempre roda: é ela que descobre arquivos novos
        bronze = registro[PIPELINE[0]]
        novos = bronze.get("resumo", {}).get("arquivos_gravados", 0)

        # Silver e Gold só trabalham se há o que processar; sem novidade, o ciclo termina em segundos
        rodar_resto = bronze["status"] == "OK" and (SEMPRE_RODAR_TUDO or novos or refazer)
        for notebook in PIPELINE[1:]:
            if not rodar_resto:
                registro[notebook] = {"status": "PULADO", "duracao_s": 0}
                continue
            registro[notebook] = executar(notebook)
            if registro[notebook]["status"] == "ERRO":
                break                                          # etapas seguintes dependem desta
        refazer = any(registro.get(n, {}).get("status") == "ERRO" for n in PIPELINE)   # erro: o próximo ciclo roda tudo de novo
        sem_dados = 0 if novos else sem_dados + 1
        historico.append(registro)
        escrever_metadata(ANOTACAO_LOG, {"inicio_utc": INICIO.isoformat(), "pipeline": PIPELINE, "ciclos": historico})

        detalhe = (f"Bronze: {novos} arquivos · {bronze['resumo'].get('linhas_gravadas', 0)} linhas · {bronze['resumo'].get('rejeitados', 0)} rejeitados"
                   if bronze["status"] == "OK" else f"❌ Bronze: {bronze['erro'][:120]}")
        silver = registro.get("./04_ingestao_silver")
        if silver and silver["status"] == "PULADO":
            detalhe += " | Silver/Gold: sem novidades"
            silver = None
        if silver:
            detalhe += (f" | Silver: {silver['resumo'].get('aprovadas', 0)} aprovadas · {silver['resumo'].get('quarentena', 0)} quarentena"
                        if silver["status"] == "OK" else f" | ❌ Silver: {silver['erro'][:120]}")
        gold = registro.get("./05_ingestao_gold")
        if gold and gold["status"] != "PULADO":
            detalhe += (f" | Gold: {gold['resumo'].get('alertas_novos', 0)} alertas novos"
                        if gold["status"] == "OK" else f" | ❌ Gold: {gold['erro'][:120]}")
        duracao = round(sum(registro[n]["duracao_s"] for n in PIPELINE if n in registro), 1)
        print(f"[{datetime.now(timezone.utc):%H:%M:%S}] ciclo {ciclo:>3} · {detalhe} · {duracao} s · sem dados: {sem_dados}/{MAX_CICLOS_SEM_DADOS}")
        for notebook in PIPELINE:                              # erro novo: mostra o texto completo uma vez, logo abaixo do ciclo
            resultado = registro.get(notebook, {})
            if resultado.get("status") == "ERRO" and erros_mostrados.get(notebook) != resultado["erro"]:
                erros_mostrados[notebook] = resultado["erro"]
                print(f"    ↳ {notebook}: {resultado['erro_completo']}")
        if sem_dados < MAX_CICLOS_SEM_DADOS:
            time.sleep(INTERVALO_S)
except KeyboardInterrupt:
    print("⏹ Interrompido manualmente (o histórico até aqui está salvo em metadata/)")
finally:
    destravar()                                  # libera a trava mesmo em erro ou cancelamento

motivo_fim = ("sem dados novos" if sem_dados >= MAX_CICLOS_SEM_DADOS else "limite de ciclos" if ciclo >= MAX_CICLOS else "interrupção manual")
print(f"Fim: {ciclo} ciclos · motivo: {motivo_fim}")

✅ Trava obtida: só esta execução processa (orquestrador_20261008_194726)
Início: 2026-10-08 19:47:26 UTC
[19:49:19] ciclo   1 · Bronze: 8 arquivos · 40 linhas · 0 rejeitados | Silver: 26 aprovadas · 14 quarentena | Gold: 16 alertas novos · 108.5 s · sem dados: 0/5
[19:52:05] ciclo   2 · Bronze: 2 arquivos · 27 linhas · 0 rejeitados | Silver: 14 aprovadas · 13 quarentena | Gold: 6 alertas novos · 105.4 s · sem dados: 0/5
[19:54:41] ciclo   3 · Bronze: 1 arquivos · 6 linhas · 0 rejeitados | Silver: 5 aprovadas · 1 quarentena | Gold: 2 alertas novos · 96.1 s · sem dados: 0/5
[19:57:10] ciclo   4 · Bronze: 1 arquivos · 1 linhas · 0 rejeitados | Silver: 1 aprovadas · 0 quarentena | Gold: 0 alertas novos · 87.9 s · sem dados: 0/5
[19:59:33] ciclo   5 · Bronze: 1 arquivos · 1 linhas · 0 rejeitados | Silver: 0 aprovadas · 1 quarentena | Gold: 2 alertas novos · 83.3 s · sem dados: 0/5
[20:02:10] ciclo   6 · Bronze: 1 arquivos · 1 linhas · 0 rejeitados | Silver: 1 aprovadas · 0 quarentena | Gold

## Etapa 4 — Relatório da execução
**Faz:** monta a tabela dos ciclos e confere a Bronze inteira ao final.
**Valida:** nenhum ciclo com erro, todos os arquivos do `raw` na Bronze (ou anotados como rejeitados), nenhum arquivo ingerido mais de uma vez (regra 2). `id_pedido` repetido na Silver é só monitorado (⚠️): a planilha não pede deduplicação por pedido.

In [0]:
etapa_bronze = PIPELINE[0]
def campo(r, etapa, chave):
    """Valor do resumo de uma etapa num ciclo (0 se a etapa não rodou ou falhou)."""
    return r.get(etapa, {}).get("resumo", {}).get(chave, 0)

relatorio = pd.DataFrame([{
    "ciclo": r["ciclo"], "inicio_utc": r["inicio_utc"],
    "status": " / ".join(r[e]["status"] for e in PIPELINE if e in r),
    "bronze_arquivos": campo(r, etapa_bronze, "arquivos_gravados"), "bronze_linhas": campo(r, etapa_bronze, "linhas_gravadas"),
    "bronze_rejeitados": campo(r, etapa_bronze, "rejeitados"),
    "silver_aprovadas": campo(r, "./04_ingestao_silver", "aprovadas"), "silver_quarentena": campo(r, "./04_ingestao_silver", "quarentena"),
    "gold_alertas_novos": campo(r, "./05_ingestao_gold", "alertas_novos"),
    "duracao_s": round(sum(r[e]["duracao_s"] for e in PIPELINE if e in r), 1),
    "erro": " | ".join(r[e].get("erro", "") for e in PIPELINE if e in r and r[e]["status"] == "ERRO")} for r in historico])

bronze = ler_delta(CAMINHO_BRONZE)
por_arquivo = bronze.groupBy("bronze_source_file").agg(F.countDistinct("bronze_ingested_at").alias("ingestoes"))
arquivos_bronze = {r["bronze_source_file"] for r in por_arquivo.select("bronze_source_file").collect()}
faltando = set(listar_arquivos_origem()) - arquivos_bronze - set(ler_metadata("bronze_ignorados") or {})

# ------------------------------ validação ------------------------------
checar(not relatorio["erro"].astype(bool).any(), "Nenhum ciclo com erro", f"{int(relatorio['erro'].astype(bool).sum())} ciclos com erro", parar=False)
checar(not faltando, "Todo arquivo do raw está na Bronze ou anotado como rejeitado",
       f"{len(faltando)} arquivos ainda não ingeridos (chegaram após o último ciclo?)", parar=False)
checar(por_arquivo.filter(F.col("ingestoes") > 1).count() == 0, "Nenhum arquivo ingerido mais de uma vez (regra 2)",
       "Há arquivos ingeridos mais de uma vez")
if tabela_delta_existe(CAMINHO_SILVER):
    checar(ler_delta(CAMINHO_SILVER).groupBy("id_pedido").count().filter("count > 1").count() == 0,
           "Nenhum id_pedido repetido na Silver", "Há id_pedido repetidos na Silver (fora da planilha: só monitorado)", parar=False)

# ------------------------------ resultado ------------------------------
display(relatorio) if len(relatorio) else print("Nenhum ciclo concluído.")   # display() não aceita tabela vazia
print(f"Ciclos: {len(relatorio)} · arquivos na Bronze: {int(relatorio['bronze_arquivos'].sum())} · linhas: {int(relatorio['bronze_linhas'].sum())} · "
      f"Silver: {int(relatorio['silver_aprovadas'].sum())} aprovadas, {int(relatorio['silver_quarentena'].sum())} quarentena · "
      f"Gold: {int(relatorio['gold_alertas_novos'].sum())} alertas · "
      f"Bronze total: {len(arquivos_bronze)} micro-lotes · registro: {CONTAINER_SQUAD}/{caminho_metadata(ANOTACAO_LOG)}")

✅ Nenhum ciclo com erro
✅ Todo arquivo do raw está na Bronze ou anotado como rejeitado
✅ Nenhum arquivo ingerido mais de uma vez (regra 2)
✅ Nenhum id_pedido repetido na Silver


ciclo,inicio_utc,status,bronze_arquivos,bronze_linhas,bronze_rejeitados,silver_aprovadas,silver_quarentena,gold_alertas_novos,duracao_s,erro
1,2026-10-08T19:47:31+00:00,OK / OK / OK,8,40,0,26,14,16,108.5,
2,2026-10-08T19:50:20+00:00,OK / OK / OK,2,27,0,14,13,6,105.4,
3,2026-10-08T19:53:05+00:00,OK / OK / OK,1,6,0,5,1,2,96.1,
4,2026-10-08T19:55:42+00:00,OK / OK / OK,1,1,0,1,0,0,87.9,
5,2026-10-08T19:58:10+00:00,OK / OK / OK,1,1,0,0,1,2,83.3,
6,2026-10-08T20:00:33+00:00,OK / OK / OK,1,1,0,1,0,1,97.0,
7,2026-10-08T20:03:11+00:00,OK / PULADO / PULADO,0,0,0,0,0,0,2.1,
8,2026-10-08T20:04:13+00:00,OK / PULADO / PULADO,0,0,0,0,0,0,4.2,
9,2026-10-08T20:05:17+00:00,OK / PULADO / PULADO,0,0,0,0,0,0,2.0,
10,2026-10-08T20:06:20+00:00,OK / PULADO / PULADO,0,0,0,0,0,0,2.0,


Ciclos: 11 · arquivos na Bronze: 14 · linhas: 76 · Silver: 47 aprovadas, 29 quarentena · Gold: 27 alertas · Bronze total: 353 micro-lotes · registro: squad2/metadata/ecommerce_pedidos/orquestrador_20261008_194726.json
